# Exercise 7: Beyond NLLB's languages

*Machine Translation · version 0.1.0*

> NLLB-200 does not cover Nogai, Kumyk or Sakha, but TurkicNLP's LLM backend accepts all 24 languages. Take 20 Nogai (or Sakha) sentences with reference translations into Russian, written or checked by a native speaker. Translate them with `Pipeline("nog", processors=["tokenize", "translate"], translate_backend="llm", translate_tgt_lang="rus", ...)` using two models of different sizes, and compute chrF against the references. Then reverse the task with a short prompt to the same models (Russian → Nogai) and run `turkicnlp.LanguageDetection()` on the outputs. How often is the "Nogai" output labelled as Kazakh, Kyrgyz or another neighbour (see the language-identification section in chapter 4)?

**Setup.** `pip install "turkicnlp[hfst]==0.5.0" torch transformers sentencepiece sacrebleu requests` (the compiled `hfst` package needs Python 3.11 or 3.12; elsewhere install `pyhfst`). NLLB-200 distilled 600M is loaded from `NLLB_PATH` (a local copy) or from the Hugging Face Hub at the revision pinned in the book; it runs in float32 on the CPU, about 1.7 s per sentence on four cores. Translations are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so an interrupted run resumes where it stopped. Test data come from the commit-pinned TIL release (X-WMT test sets, MIT licence); the X-WMT files are tokenised, and the code removes the spaces around punctuation before scoring (a heuristic, see `detok`). Scores use SacreBLEU with its default settings: BLEU with the `13a` tokeniser, chrF, and chrF++ (`word_order=2`).

**Inputs.** This exercise needs two things the book cannot ship: native-speaker reference translations and an LLM server. Provide a tab-separated file of Nogai sentences and Russian references in `NOGAI_REFS`, the model names in `LLM_MODELS` (comma-separated) and the server in `LLM_BASE_URL` and `LLM_API_KEY`. The language-ID model is GlotLID through `turkicnlp.LanguageDetection()`; its labels are unrestricted, so an English or Russian answer is labelled as such and not forced into a Turkic label.

In [ ]:
import collections, hashlib, os, re, statistics, time
from pathlib import Path
import requests

# Commit-pinned TIL release files (X-WMT test sets and the corpus README); MIT licence.
TIL = "https://raw.githubusercontent.com/turkic-interlingua/til-mt/7d988cc47dba889d02262d893bf9a46e43a6bb36/"
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
FETCHED = {}                                   # url -> sha256 of every file read (saved with the outputs)

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def detok(s):
    """Undo the X-WMT tokenisation (spaces around punctuation) so references match raw MT output."""
    s = re.sub(r" ([’'] ?)", lambda m: m.group(1).strip(), s)       # Sialkot ’ taki -> Sialkot’taki
    s = re.sub(r" ([.,!?;:%)\]»])", r"\1", s)
    return re.sub(r"([(\[«]) ", r"\1", s).strip()

def xwmt(src, tgt, n=None):
    """The first n sentence pairs of the X-WMT test set for src-tgt (X-WMT language codes)."""
    s = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{src}").read_text(encoding="utf-8").splitlines()
    t = fetch(f"{TIL}xwmt/test/{src}-{tgt}/{src}-{tgt}.{tgt}").read_text(encoding="utf-8").splitlines()
    assert len(s) == len(t), (src, tgt)
    pairs = [(detok(a), detok(b)) for a, b in zip(s, t) if a.strip() and b.strip()]
    return pairs[:n] if n else pairs

NLLB_CODE = {"az": "azj_Latn", "ba": "bak_Cyrl", "kk": "kaz_Cyrl", "ky": "kir_Cyrl", "tr": "tur_Latn",
             "uz": "uzn_Latn", "en": "eng_Latn", "ru": "rus_Cyrl"}
NLLB = {}

def nllb():
    """NLLB-200 distilled 600M: from NLLB_PATH (a local copy) or the pinned Hub revision; float32, CPU."""
    if not NLLB:
        import torch
        from transformers import AutoModelForSeq2SeqLM, NllbTokenizer
        path = os.environ.get("NLLB_PATH", "facebook/nllb-200-distilled-600M")
        rev = None if Path(path).exists() else "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
        NLLB["tok"] = NllbTokenizer.from_pretrained(path, revision=rev)
        NLLB["model"] = AutoModelForSeq2SeqLM.from_pretrained(path, revision=rev, torch_dtype=torch.float32).eval()
        NLLB["torch"] = torch
        torch.set_num_threads(os.cpu_count() or 1)
    return NLLB["tok"], NLLB["model"]

CACHE_FILE = CACHE / "nllb-translations.jsonl"       # translations are cached, so a run can resume
LAST_SECONDS = 0.0

def _cached():
    import json
    out = {}
    if CACHE_FILE.exists():
        for line in CACHE_FILE.read_text(encoding="utf-8").splitlines():
            r = json.loads(line); out[tuple(r["key"])] = (r["out"], r["capped"], r["sec"])
    return out

def translate(sents, src, tgt, beams=1, batch=8, max_new=256):
    """Translate with explicit source and target language tokens (greedy unless beams > 1).
    Returns (outputs, number of outputs that reached the generation ceiling without an
    end-of-sequence token). Results are cached per sentence in CACHE_FILE, with the generation
    time of their batch divided by its size; LAST_SECONDS holds the total for the last call."""
    global LAST_SECONDS
    import json
    cache = _cached()
    key = lambda s: [src, tgt, beams, max_new, s]
    todo = [s for s in dict.fromkeys(sents) if tuple(key(s)) not in cache]
    if todo:
        tok, model = nllb()
        tok.src_lang = NLLB_CODE[src]
        CACHE_FILE.parent.mkdir(parents=True, exist_ok=True)
        with NLLB["torch"].inference_mode(), CACHE_FILE.open("a", encoding="utf-8") as f:
            for i in range(0, len(todo), batch):
                chunk = todo[i:i + batch]
                t0 = time.time()
                enc = tok(chunk, return_tensors="pt", padding=True)
                gen = model.generate(**enc, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB_CODE[tgt]),
                                     num_beams=beams, do_sample=False, max_new_tokens=max_new)
                sec = (time.time() - t0) / len(chunk)
                for s, row, text in zip(chunk, gen, tok.batch_decode(gen, skip_special_tokens=True)):
                    capped = int(row[-1] != tok.eos_token_id and row[-1] != tok.pad_token_id)
                    cache[tuple(key(s))] = (text, capped, sec)
                    f.write(json.dumps({"key": key(s), "out": text, "capped": capped, "sec": sec}, ensure_ascii=False) + "\n")
                f.flush()
    res = [cache[tuple(key(s))] for s in sents]
    LAST_SECONDS = sum(r[2] for r in res)
    return [r[0] for r in res], sum(r[1] for r in res)

def metrics(hyps, refs):
    """Corpus BLEU (13a), chrF and chrF++ with SacreBLEU, rounded, plus the chrF++ signature."""
    from sacrebleu.metrics import BLEU, CHRF
    bleu, chrf, chrfpp = BLEU(tokenize="13a"), CHRF(), CHRF(word_order=2)
    r = {"BLEU": bleu.corpus_score(hyps, [refs]).score, "chrF": chrf.corpus_score(hyps, [refs]).score,
         "chrF++": chrfpp.corpus_score(hyps, [refs]).score}
    return {k: round(v, 2) for k, v in r.items()}, str(chrfpp.get_signature())

In [ ]:
# NLLB-200 has no Nogai, Kumyk or Sakha; TurkicNLP's LLM backend accepts them. This cell runs only
# when an OpenAI-compatible server and reference translations are provided.
REFS = os.environ.get("NOGAI_REFS")                      # TSV: Nogai sentence <TAB> Russian reference
MODELS = [m for m in os.environ.get("LLM_MODELS", "").split(",") if m]
if not (REFS and MODELS):
    print("Not configured: set NOGAI_REFS (a TSV of Nogai sentences and Russian references), LLM_MODELS, "
          "LLM_BASE_URL and LLM_API_KEY.")
else:
    import turkicnlp
    rows = [l.split("\t") for l in Path(REFS).read_text(encoding="utf-8").splitlines() if "\t" in l]
    lid = turkicnlp.LanguageDetection()
    for model in MODELS:
        nlp = turkicnlp.Pipeline("nog", processors=["tokenize", "translate"], translate_backend="llm",
                                 translate_tgt_lang="rus", llm_model=model, llm_base_url=os.environ.get("LLM_BASE_URL"),
                                 llm_api_key=os.environ.get("LLM_API_KEY", "none"))
        hyps = [nlp(src).translation for src, _ in rows]
        print(model, "Nogai->Russian", metrics(hyps, [ref for _, ref in rows])[0])
        back = turkicnlp.Pipeline("rus", processors=["tokenize", "translate"], translate_backend="llm",
                                  translate_tgt_lang="nog", llm_model=model, llm_base_url=os.environ.get("LLM_BASE_URL"),
                                  llm_api_key=os.environ.get("LLM_API_KEY", "none"))
        labels = collections.Counter(lid.model.predict([back(ref).translation.replace("\n", " ")], k=1)[0][0][0]
                                     for _, ref in rows)
        print(model, "Russian->'Nogai' outputs by language-ID label:", labels.most_common())

## Worked solution

Output of `experiments/ch10_exercises/run_ch10_exercises.py`:

```
Not configured: set NOGAI_REFS (a TSV of Nogai sentences and Russian references), LLM_MODELS, LLM_BASE_URL and LLM_API_KEY.
```

The exercise was not run in the environment used for the worked solution: there was no LLM server and no native-speaker Nogai–Russian reference set. When you run it, report chrF for each model and direction, and the distribution of language-ID labels for the "Nogai" outputs. A high share of other language labels flags possible language confusion; it does not prove that the translations are in those languages. Check a sample by hand, because GlotLID itself can confuse Nogai with its neighbours on short sentences.

## Your turn — extension

1. Run the cell with a small and a large model.
2. Repeat for Sakha, where the nearest NLLB language (by script and family) is much more distant than Kazakh is from Nogai.